# Notebook 1 (CPU — 0 GPU Hours): Pre-Download Model & PrismML Binary (Kaggle & Google Colab + Keep-Alive)

Run this notebook **on CPU only** (*Accelerator: None*, *Internet: ON*) so you do not spend GPU quota waiting for large file downloads. It prepares a reusable bundle containing:
1. Prebuilt `llama-server` CUDA 12.8 binary (`prism-b10828-a153697`)
2. Main model `Ternary-Bonsai-2-27B-Abliterated-v2-PQ2_0-MTP.gguf` (7.66 GB)
3. *(Optional)* Vision projector `Ternary-Bonsai-2-27B-mmproj-Q8_0.gguf` (0.63 GB) for image understanding

---

## Step-by-Step Guide: What You Should Do

### Option A — On Kaggle
1. Open **Session options** (right sidebar): set **Accelerator** to **None** (CPU) and turn **Internet** **ON**.
2. Review the configuration variables in the code cell below (the default values work out of the box).
3. Click **Save Version** → **Save & Run All** (keeps `KEEP_ALIVE_HOLD_AFTER_DONE = False` so the version finishes cleanly once downloads complete).
4. After the run finishes, open the notebook's **Output** tab and save `/kaggle/working/bonsai_bundle` as a **Private Dataset** (or attach the notebook output directly).
5. Open **Notebook 2 (GPU)** and attach that Dataset via **Add Input** → **Your Datasets**.

### Option B — On Google Colab
1. Go to **Runtime** → **Change runtime type** → select **CPU**.
2. Run the code cell below and approve the Google Drive authorization prompt when it appears.
3. The script automatically saves all files to `/content/drive/MyDrive/bonsai_bundle`.
4. Open **Notebook 2 (GPU)** and set `COLAB_USE_GOOGLE_DRIVE = True` in Cell 1 so it loads everything directly from Drive without re-downloading.

---

## Configuration Variables & Functions Reference

### 1. Model Download & Storage Variables
| Variable | Default | What It Does & When to Change It |
| :--- | :--- | :--- |
| `PLATFORM` | *(auto)* | Auto-detects `'kaggle'`, `'colab'`, or `'local'` to choose the right storage directory. |
| `REPO_ID` | `BoldingBuilds/Ternary-Bonsai-2-27B-Abliterated-v2-PQ2_0-MTP-GGUF` | Hugging Face repository hosting the GGUF model files. |
| `PRISM_RELEASE_TAG` | `prism-b10828-a153697` | GitHub release tag of `PrismML-Eng/llama.cpp` containing the prebuilt CUDA 12.8 `llama-server` binary. |
| `INCLUDE_PLAIN_NO_MTP` | `False` | Set to `True` if you also want to download the standard non-MTP model (`7.21 GB`) as a fallback in case MTP causes issues with image inputs. |
| `INCLUDE_VISION` | `True` | Downloads `Ternary-Bonsai-2-27B-mmproj-Q8_0.gguf` (`0.63 GB`) so you can enable or disable Vision freely in Notebook 2. |
| `BUNDLE_DIR` | *(auto)* | Target folder for `prism-bin/` and `models/` (`/kaggle/working/bonsai_bundle` on Kaggle, `/content/drive/MyDrive/bonsai_bundle` on Colab, `./bonsai_bundle` locally). |

### 2. Keep-Alive Variables
| Variable | Default | What It Does & When to Change It |
| :--- | :--- | :--- |
| `ENABLE_KEEP_ALIVE` | `True` | Keeps the notebook session active during large downloads to prevent idle disconnects. |
| `KEEP_ALIVE_MIN_HOURS` | `5.0` | Minimum keep-alive duration in hours (used by the background heartbeat and optional post-download hold). |
| `KEEP_ALIVE_PING_INTERVAL` | `45` | Interval (in seconds) between background heartbeat writes to `/tmp/.keepalive_heartbeat`. |
| `KEEP_ALIVE_HOLD_AFTER_DONE` | `False` | `False`: exits immediately once downloads finish (required for Kaggle **Save & Run All**). `True`: keeps the cell running after downloads finish until `KEEP_ALIVE_MIN_HOURS` elapses. |
| `KEEP_ALIVE_ENABLE_BROWSER_JS` | `True` | Injects lightweight browser JavaScript to dismiss idle/reconnect popups on Colab and Kaggle. |

### 3. Built-in Helper Functions
- **`inject_browser_keepalive()`**: Injects a browser-side timer (every 30s) that simulates light mouse movement and clicks reconnect/idle confirmation buttons on Colab/Kaggle.
- **`_bg_keepalive_worker()`**: Background thread that periodically writes a timestamp to `/tmp/.keepalive_heartbeat` while large files are downloading so kernel I/O stays active.


In [ ]:
!pip install -q hf_transfer huggingface_hub

import datetime
import glob
import json
import os
import shutil
import sys
import tarfile
import threading
import time
import urllib.request

# Auto-detect execution platform: 'kaggle', 'colab', or 'local'
if os.path.exists("/kaggle/working"):
    PLATFORM = "kaggle"
elif "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content"):
    PLATFORM = "colab"
else:
    PLATFORM = "local"

# Enable Rust-based hf_transfer for much faster Hugging Face downloads
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import hf_hub_download

# ==========================================
# 1. MODEL DOWNLOAD & STORAGE CONFIGURATION
# ==========================================
REPO_ID = "BoldingBuilds/Ternary-Bonsai-2-27B-Abliterated-v2-PQ2_0-MTP-GGUF"
PRISM_RELEASE_TAG = "prism-b10828-a153697"
INCLUDE_PLAIN_NO_MTP = False  # Set True to also download the non-MTP variant (7.21 GB)
INCLUDE_VISION = True         # Set True to pre-download mmproj-Q8_0 (0.63 GB) for Vision support

# Automatically pick storage directory based on platform (you can override manually if needed)
if PLATFORM == "kaggle":
    BUNDLE_DIR = "/kaggle/working/bonsai_bundle"
elif PLATFORM == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    BUNDLE_DIR = "/content/drive/MyDrive/bonsai_bundle"
else:
    BUNDLE_DIR = "./bonsai_bundle"

# ==========================================
# 2. AUTO KEEP-ALIVE CONFIGURATION
# ==========================================
ENABLE_KEEP_ALIVE = True             # True: prevent idle disconnects during download (and optionally after)
KEEP_ALIVE_MIN_HOURS = 5.0           # Minimum duration (hours) to keep the session alive
KEEP_ALIVE_PING_INTERVAL = 45        # Heartbeat interval (seconds)
KEEP_ALIVE_HOLD_AFTER_DONE = False   # False: exit immediately after download (best for Kaggle Save & Run All);
                                     # True: keep holding the session until KEEP_ALIVE_MIN_HOURS is reached
KEEP_ALIVE_ENABLE_BROWSER_JS = True  # Inject browser JS to auto-dismiss idle popups on Colab & Kaggle

BIN_DIR = os.path.join(BUNDLE_DIR, "prism-bin")
MODEL_DIR = os.path.join(BUNDLE_DIR, "models")
os.makedirs(BIN_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)


def inject_browser_keepalive():
    """Inject JavaScript into the notebook tab to prevent browser idle timeouts on Colab/Kaggle."""
    try:
        from IPython.display import Javascript, display
        js_code = """
        (function() {
            if (window.__keepAliveInjected) return;
            window.__keepAliveInjected = true;
            setInterval(function() {
                const colabBtn = document.querySelector('colab-connect-button') || document.querySelector('#top-toolbar colab-connect-button');
                if (colabBtn && colabBtn.shadowRoot) {
                    const inner = colabBtn.shadowRoot.querySelector('#connect');
                    if (inner && inner.innerText && inner.innerText.toLowerCase().includes('reconnect')) {
                        inner.click();
                    }
                }
                document.querySelectorAll('mwc-button, paper-button, button').forEach(b => {
                    const txt = (b.innerText || b.textContent || '').trim().toLowerCase();
                    if (txt === 'ok' || txt === 'reconnect' || txt === 'keep working' || txt === 'stay connected') {
                        b.click();
                    }
                });
                document.dispatchEvent(new MouseEvent('mousemove', {bubbles: true, clientX: Math.random()*100, clientY: Math.random()*100}));
            }, 30000);
        })();
        """
        display(Javascript(js_code))
    except Exception:
        pass


_stop_bg_event = threading.Event()
_bg_start_ts = time.time()


def _bg_keepalive_worker():
    """Write a lightweight heartbeat file in /tmp periodically to keep kernel I/O active during large downloads."""
    target_sec = int(KEEP_ALIVE_MIN_HOURS * 3600)
    while not _stop_bg_event.is_set():
        elapsed = time.time() - _bg_start_ts
        if elapsed >= target_sec:
            break
        try:
            with open("/tmp/.keepalive_heartbeat", "w") as f:
                f.write(str(time.time()))
        except Exception:
            pass
        _stop_bg_event.wait(KEEP_ALIVE_PING_INTERVAL)


if ENABLE_KEEP_ALIVE:
    if KEEP_ALIVE_ENABLE_BROWSER_JS:
        inject_browser_keepalive()
    bg_thread = threading.Thread(target=_bg_keepalive_worker, daemon=True)
    bg_thread.start()
    print(f"[KEEP-ALIVE] Anti-idle protection active on {PLATFORM.upper()} (target: {KEEP_ALIVE_MIN_HOURS}h).")

# 1. Download & extract prebuilt CUDA 12.8 llama-server binary
api_url = f"https://api.github.com/repos/PrismML-Eng/llama.cpp/releases/tags/{PRISM_RELEASE_TAG}"
req = urllib.request.Request(api_url, headers={"User-Agent": "cloud-cpu-prep"})
with urllib.request.urlopen(req, timeout=20) as resp:
    rel_data = json.loads(resp.read().decode("utf-8"))

asset = next(
    (a for a in rel_data.get("assets", []) if "bin-linux-cuda-12.8-x64.tar.gz" in a["name"]),
    None,
)
if not asset:
    raise RuntimeError("Could not find the cuda-12.8 Linux x64 archive in the specified GitHub release.")

tar_path = os.path.join("/tmp", asset["name"])
print(f"Downloading {asset['name']}...")
urllib.request.urlretrieve(asset["browser_download_url"], tar_path)

with tarfile.open(tar_path, "r:gz") as tf:
    tf.extractall(BIN_DIR)
os.remove(tar_path)
print("Extracted prism-bin successfully!")

# 2. Download GGUF model files from Hugging Face
files_to_download = ["Ternary-Bonsai-2-27B-Abliterated-v2-PQ2_0-MTP.gguf"]
if INCLUDE_PLAIN_NO_MTP:
    files_to_download.append("Ternary-Bonsai-2-27B-Abliterated-v2-PQ2_0.gguf")
if INCLUDE_VISION:
    files_to_download.append("Ternary-Bonsai-2-27B-mmproj-Q8_0.gguf")

for fname in files_to_download:
    print(f"Downloading {fname}...")
    hf_hub_download(repo_id=REPO_ID, filename=fname, local_dir=MODEL_DIR)

# Clean up hidden Hugging Face cache folder so the Dataset/Drive bundle stays compact
hf_cache = os.path.join(MODEL_DIR, ".cache")
if os.path.exists(hf_cache):
    shutil.rmtree(hf_cache)

print("=" * 60)
print(f"COMPLETED ON {PLATFORM.upper()}! Files saved in {BUNDLE_DIR}:")
for p in sorted(glob.glob(f"{BUNDLE_DIR}/**/*", recursive=True)):
    if os.path.isfile(p) and ("gguf" in p or "llama-server" in p):
        size_mb = os.path.getsize(p) / (1024 * 1024)
        print(f" - {p} ({size_mb:.1f} MB)")
print("=" * 60)
if PLATFORM == "kaggle":
    print("Next step: Save this notebook's Output as a Private Dataset, then attach it to Notebook 2 (GPU).")
else:
    print("Next step: Set COLAB_USE_GOOGLE_DRIVE = True in Notebook 2 (GPU) to load directly from Google Drive.")

# Optional: keep session alive after downloads complete if KEEP_ALIVE_HOLD_AFTER_DONE = True
if ENABLE_KEEP_ALIVE and KEEP_ALIVE_HOLD_AFTER_DONE:
    target_sec = int(KEEP_ALIVE_MIN_HOURS * 3600)
    print(f"\n[KEEP-ALIVE] Holding session active until {KEEP_ALIVE_MIN_HOURS} hours elapsed...")
    try:
        while (time.time() - _bg_start_ts) < target_sec:
            hrs_done = (time.time() - _bg_start_ts) / 3600.0
            hrs_left = max(0.0, (target_sec - (time.time() - _bg_start_ts)) / 3600.0)
            sys.stdout.write(
                f"\r[{datetime.datetime.now().strftime('%H:%M:%S')}] "
                f"Keep-Alive: {hrs_done:.2f}h / {KEEP_ALIVE_MIN_HOURS:.1f}h ({hrs_left:.2f}h left)   "
            )
            sys.stdout.flush()
            time.sleep(KEEP_ALIVE_PING_INTERVAL)
    except KeyboardInterrupt:
        print("\n[KEEP-ALIVE] Stopped by user.")

_stop_bg_event.set()
